In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score # Removed trailing comma
import zipfile
import os

zip_file = "archive.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset Extracted Successfully!")

for root, dirs, files in os.walk("dataset"):
    for file in files:
        print(os.path.join(root, file))

coursera = pd.read_csv("dataset/Coursera.csv")
udemy = pd.read_csv("dataset/Udemy.csv")
edx = pd.read_csv("dataset/edx.csv")
skillshare = pd.read_csv("dataset/skillshare.csv")

print("Coursera:", coursera.shape)
print("Udemy:", udemy.shape)
print("edX:", edx.shape)
print("Skillshare:", skillshare.shape)
print(coursera.head())

print(coursera.columns)
print(udemy.columns)
print(edx.columns)
print(skillshare.columns)

for df in [coursera, udemy, edx, skillshare]:
    for col in df.select_dtypes(include=['number']).columns:
        if df[col].isnull().any():
            df[col] = df[col].astype(str).replace('nan', '')

coursera.fillna("", inplace=True)
udemy.fillna("", inplace=True)
edx.fillna("", inplace=True)
skillshare.fillna("", inplace=True)
coursera.drop_duplicates(inplace=True)
udemy.drop_duplicates(inplace=True)
edx.drop_duplicates(inplace=True)
skillshare.drop_duplicates(inplace=True)
print("Coursera:",coursera.shape)
print("Udmey:",udemy.shape)
print("edx:",edx.shape)
print("skillshare:",skillshare.shape)
print(coursera.head())
print(udemy.head())
print(edx.head())
print(skillshare.head())
print("Coursera null values:\n", coursera.isnull().sum())
print("Udemy null values:\n", udemy.isnull().sum())
print("edX null values:\n", edx.isnull().sum())
print("Skillshare null values:\n", skillshare.isnull().sum())
coursera_df = pd.DataFrame({
    "title": coursera["course"],
    "content": (
        coursera["course"].astype(str) + " " +
        coursera["skills"].astype(str) + " " +
        coursera["level"].astype(str)
    ),
    "platform": "Coursera",
    "rating": coursera["rating"]
})

udemy_df = pd.DataFrame({
    "title": udemy["title"],
    "content": (
        udemy["title"].astype(str) + " " +
        udemy["description"].astype(str) + " " +
        udemy["level"].astype(str)
    ),
    "platform": "Udemy",
    "rating": udemy["rating"]
})

edx_df = pd.DataFrame({
    "title": edx["title"],
    "content": (
        edx["title"].astype(str) + " " +
        edx["subject"].astype(str) + " " +
        edx["associatedskills"].astype(str) + " " +
        edx["level"].astype(str)
    ),
    "platform": "edX",
    "rating": 0
})

skillshare_df = pd.DataFrame({
    "title": skillshare["title"],
    "content": (
        skillshare["title"].astype(str) + " " +
        skillshare["instructor"].astype(str)
    ),
    "platform": "Skillshare",
    "rating": 0
})

courses = pd.concat(
    [coursera_df, udemy_df, edx_df, skillshare_df],
    ignore_index=True
)

print("Total Courses:", len(courses))
print(courses.head())
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000
)

tfidf_matrix = vectorizer.fit_transform(
    courses["content"]
)

print(tfidf_matrix.shape)
from sklearn.metrics.pairwise import cosine_similarity


def recommend_courses(user_interest, top_n=10):

    user_vector = vectorizer.transform([user_interest])

    similarity_scores = cosine_similarity(
        user_vector,
        tfidf_matrix
    ).flatten()

    top_indices = similarity_scores.argsort()[::-1][:top_n]

    recommendations = courses.iloc[top_indices].copy()

    recommendations["Similarity"] = similarity_scores[top_indices]

    return recommendations[
        ["title", "platform", "rating", "Similarity"]
    ]
print(recommend_courses(
    "Machine Learning Python AI",
    10
))
print(recommend_courses("Data Science"))
print(recommend_courses("Web Development"))
print(recommend_courses("Cyber Security"))
print(recommend_courses("Artificial Intelligence"))
while True:

    query = input("Enter Interest (or exit): ")

    if query.lower() == "exit":
        break

    result = recommend_courses(query, 10)

    print("\nRecommended Courses:\n")
    print(result.to_string(index=False))
    import matplotlib.pyplot as plt

courses["platform"].value_counts().plot(
    kind='bar'
)

plt.title("Courses per Platform")
plt.xlabel("Platform")
plt.ylabel("Count")
plt.show()


courses['rating'] = pd.to_numeric(courses['rating'], errors='coerce')
courses['rating'] = courses['rating'].fillna(0)

courses.groupby("platform")["rating"].mean().plot(
    kind='bar'
)
plt.title("Average Rating per platform")
plt.xlabel("platform")
plt.ylabel("Average rating")
plt.show()

top_courses = courses.sort_values(
    by="rating",
    ascending=False
).head(10)
print(top_courses[["title","platform","rating"]])
import numpy as np

def evaluate_recommendation(query, relevant_keywords, k=10):

    recs = recommend_courses(query, k)

    titles = recs["title"].astype(str).tolist()

    hits = 0

    for title in titles:
        for keyword in relevant_keywords:
            if keyword.lower() in title.lower():
                hits += 1
                break

    precision = hits / k


    recall = hits / len(relevant_keywords) if len(relevant_keywords) > 0 else 0

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


precision_scores = []
recall_scores = []

for k in range(1,21):

    p,r,f = evaluate_recommendation(
        "Machine Learning Python",
        [
            "Machine Learning",
            "Deep Learning",
            "Artificial Intelligence"
        ],
        k
    )

    precision_scores.append(p)
    recall_scores.append(r)

plt.figure(figsize=(8,6))

plt.plot(
    recall_scores,
    precision_scores,
    marker='o'
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision Recall Curve")

plt.grid(True)

plt.show()
from sklearn.metrics import ndcg_score

true_relevance = np.asarray([[3,2,1,0,0,0,0,0,0,0]])
pred_scores = np.asarray([[10,9,8,7,6,5,4,3,2,1]])

ndcg = ndcg_score(
    true_relevance,
    pred_scores
)

print("NDCG:", ndcg)
courses['rating'] = pd.to_numeric(courses['rating'], errors='coerce')
courses['rating'] = courses['rating'].fillna(0)

courses.groupby("platform")["rating"].mean().plot(
    kind='bar'
)
plt.title("Average Rating per platform")
plt.xlabel("platform")
plt.ylabel("Average rating")
plt.show()
plt.figure(figsize=(10, 6))
sns.histplot(courses['rating'], bins=10, kde=True)
plt.title('Distribution of Course Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Courses')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
print('Top 5 Highest Rated Courses per Platform:')
for platform in courses['platform'].unique():
    print(f"\n--- {platform} ---")
    top_5_courses = courses[courses['platform'] == platform].sort_values(by='rating', ascending=False).head(5)
    display(top_5_courses[['title', 'platform', 'rating']])